# Adding a trust layer to Claude agent loops

Claude can reason about tools, but production agent loops still need deterministic controls outside the model. This notebook shows a guardrails-as-code pattern: let Claude propose useful work, then validate tool calls, enforce policy, and emit an inspectable trace before anything consequential executes. It extends the agent-loop ideas in Anthropic's [Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents) post with a small middleware layer. Pramagent is used as the concrete implementation, but the pattern is the important part: the LLM is not the final authority.

In [ ]:
%pip install -q "anthropic>=0.87.0" "pramagent>=0.8.0"

In [ ]:
import asyncio
import json
import os
from pprint import pprint

import anthropic
from pramagent import Pramagent, Verdict
from pramagent.layers import ComplianceLayer, HITLLayer, ReliabilityLayer, Rule, SafetyLayer
from pramagent.layers import ToolGuardLayer, ToolPolicy
from pramagent.layers.tool_guard import SideEffect
from pramagent.providers import AnthropicProvider

MODEL = "claude-haiku-4-5"
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise RuntimeError("Set ANTHROPIC_API_KEY in your environment before running this notebook.")

client = anthropic.Anthropic(api_key=api_key)

## The baseline agent loop

Start with a tiny Claude tool loop. It has one mock side-effect tool, `send_email`. The baseline lets Claude choose the tool and arguments, then the application executes the proposed tool directly. That is useful for a demo, but it leaves several production gaps: unvalidated tool calls, no tenant policy, no human approval gate, and no audit trace explaining why execution was allowed.

In [ ]:
tools = [
    {
        "name": "send_email",
        "description": "Send an email to a user. This is a mock tool in the notebook.",
        "input_schema": {
            "type": "object",
            "properties": {
                "to": {"type": "string"},
                "subject": {"type": "string"},
                "body": {"type": "string"},
            },
            "required": ["to", "subject", "body"],
        },
    }
]

baseline_prompt = "Send an email to sam@example.com saying the deployment report is ready."

message = client.messages.create(
    model=MODEL,
    max_tokens=400,
    tools=tools,
    messages=[{"role": "user", "content": baseline_prompt}],
)

raw_tool_uses = [block for block in message.content if block.type == "tool_use"]
print("Claude proposed tool calls:")
for tool_use in raw_tool_uses:
    print(tool_use.name)
    pprint(tool_use.input)

## Wrapping the loop with a trust layer

The trust layer sits between the model output and tool execution. Tool policy is defined as code, not as a prompt instruction. Here the model may propose `send_email`, but the middleware checks schema, side-effect class, and tenant scope first. Consequential actions escalate to human review; silence is treated as no approval.

In [ ]:
tool_guard = ToolGuardLayer(
    policies=[
        ToolPolicy(
            name="send_email",
            side_effect=SideEffect.EXTERNAL_MESSAGE,
            action=Verdict.ESCALATE,
            allowed_tenants={"internal_demo"},
            schema={
                "type": "object",
                "properties": {
                    "to": {"type": "string", "maxLength": 200},
                    "subject": {"type": "string", "maxLength": 120},
                    "body": {"type": "string", "maxLength": 2000},
                },
                "required": ["to", "subject", "body"],
                "additionalProperties": False,
            },
            detail="External message requires explicit approval before execution.",
        )
    ]
)

armor = Pramagent(
    provider=AnthropicProvider(model=MODEL, max_tokens=400),
    compliance=ComplianceLayer(),
    safety=SafetyLayer(
        rules=[
            Rule(
                rule_id="block_bulk_export",
                action=Verdict.BLOCK,
                pattern=r"\b(dump|export)\b.*\b(users?|accounts?|secrets?)\b",
                detail="Bulk data export is blocked before provider contact.",
            )
        ]
    ),
    reliability=ReliabilityLayer(max_concurrent=4, timeout_s=30.0),
    hitl=HITLLayer(require_approval_for=["send_email"], timeout_s=2.0),
    tool_guard=tool_guard,
    escalate_policy={"pre": "hitl", "post": "log"},
)

In [ ]:
# Same tool proposal, now checked before execution.
proposal = raw_tool_uses[0]
decision = armor.validate_tool(
    proposal.name,
    proposal.input,
    tenant_id="internal_demo",
    session_id="cookbook",
)
print("tool verdict:", decision.verdict)
print("reason:", decision.reason)

if decision.verdict == Verdict.ALLOW:
    print("Would execute mock tool now.")
elif decision.verdict == Verdict.ESCALATE:
    print("Tool execution is held for approval; no side effect runs yet.")
else:
    print("Tool execution is blocked.")

In [ ]:
# A cross-tenant attempt is rejected deterministically, regardless of model wording.
cross_tenant_decision = armor.validate_tool(
    "send_email",
    {"to": "sam@example.com", "subject": "Report", "body": "The deployment report is ready."},
    tenant_id="external_tenant",
    session_id="cookbook",
)
print("cross-tenant verdict:", cross_tenant_decision.verdict)
print("reason:", cross_tenant_decision.reason)

## Trace as a first-class response field

Now run an agent request through the full trust stack. This example asks for an email action, so Claude is called, but the final action is held by the human-in-the-loop gate. The important distinction is that this is not a model refusal; it is a deterministic application control saying the action was not executed without approval.

In [ ]:
response = await armor.run(
    "Send an email to sam@example.com saying the deployment report is ready.",
    tenant_id="internal_demo",
    session_id="cookbook-trace",
    action="send_email",
)

print("blocked:", response.blocked)
print("hitl:", response.hitl)
print("output:", response.output)
print("trace hash:", response.trace.this_hash)

In [ ]:
trace_rows = [
    {
        "layer": event.layer,
        "decision": event.decision,
        "detail": event.detail,
        "latency_ms": round(event.latency_ms, 2),
    }
    for event in response.trace.layer_events
]
pprint(trace_rows)
print("chain valid:", armor.audit.verify_chain())

## When to use this pattern

This pattern is useful when an agent can call tools, move data, contact users, modify accounts, or trigger any other side effect. It is especially helpful for RBAC-scoped agents, regulated workflows, human approval queues, and systems where an auditor needs to reconstruct why a decision was allowed, blocked, or held. The tradeoff is extra code and, for HITL paths, extra latency. Keep the policy narrow: guard the consequential paths first instead of wrapping every harmless text-only call.

## Recap and resources

- Keep model reasoning and control boundaries separate.
- Let Claude propose useful actions, but validate tool execution outside the model.
- Treat `ESCALATE` as a real state: held until approval, not silently executed.
- Emit structured traces so reviewers can inspect the decision path later.

Resources:

- [Anthropic: Building Effective Agents](https://www.anthropic.com/engineering/building-effective-agents)
- [Pramagent on GitHub](https://github.com/sriram7737/pramagent)
- [Pramagent on PyPI](https://pypi.org/project/pramagent/)
- [Implementation status](https://github.com/sriram7737/pramagent/blob/main/docs/IMPLEMENTATION_STATUS.md)